# Dense And Sparse Correspondence
In this tutorial we show how DINOv3 features can be used to establish correspondences between two objects.

### Setup

Let's start by loading some pre-requisites and checking the DINOv3 repository location:
- `local` if `DINOV3_LOCATION` environment variable was set to work with a local version of DINOv3 repository;
- `github` if the code should be loaded via torch hub.

In [ ]:
import pickle
import os
import urllib

import numpy as np
from matplotlib.patches import ConnectionPatch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from tqdm import tqdm

from transformers import AutoModel, AutoImageProcessor

print("Using Hugging Face transformers for DINOv3")

### Model Loading
We load the DINOv3 ViT-L model. Feel free to try other DINOv3 models as well!

In [ ]:
# Hugging Face model name
MODEL_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"

# Load model and processor from Hugging Face
model = AutoModel.from_pretrained(MODEL_NAME)
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model.cuda()
model.eval()

# Get model config for later use
PATCH_SIZE = model.config.patch_size
HIDDEN_DIM = model.config.hidden_size

print(f"Loaded model: {MODEL_NAME}")
print(f"Patch size: {PATCH_SIZE}, Hidden dim: {HIDDEN_DIM}")

### Data Loading
Now that we have the model set up, let's load the data. It consists of two image / mask pairs:


In [ ]:
image_left = Image.open("mercedes_1.png")
image_right = Image.open("mercedes_2.png")

image_left_uri = "https://media.architecturaldigest.com/photos/599afec1d115db3276c02c73/16:9/w_1920,c_limit/vision-mercedes-maybach-6-cabriolet-front-three-quarters-01.jpg"
image_right_uri = "https://hips.hearstapps.com/hmg-prod/images/rt-mercedes-benz-cpo-1-1531414981.jpg?crop=1.00xw:0.755xh;0,0.169xh"

def load_image_from_url(url: str) -> Image:
    with urllib.request.urlopen(url) as f:
        return Image.open(f)


# image_left = load_image_from_url(image_left_uri)
# image_right = load_image_from_url(image_right_uri)

# Extract alpha channel as mask (if available)
# The alpha channel defines transparency: 255 = opaque (foreground), 0 = transparent (background)
if image_left.mode == 'RGBA':
    mask_left = image_left.split()[-1]  # Get alpha channel
else:
    mask_left = None
    
if image_right.mode == 'RGBA':
    mask_right = image_right.split()[-1]  # Get alpha channel
else:
    mask_right = None

print(f"Left image mode: {image_left.mode}, has mask: {mask_left is not None}")
print(f"Right image mode: {image_right.mode}, has mask: {mask_right is not None}")

Let's visualize the two images:

In [ ]:
plt.figure(figsize=(12, 6), dpi=300)

plt.subplot(1, 2, 1)
plt.imshow(image_left)
plt.axis('off')
plt.title("Left Image", fontsize=12)

plt.subplot(1, 2, 2)
plt.imshow(image_right)
plt.axis('off')
plt.title("Right Image", fontsize=12)
        
plt.show()

### Data Transforms

Since our models run with a patch size of 16, we resize images such that dimensions align well with the 16x16 grid.

In [ ]:
IMAGE_SIZE = 768

# quantization filter for the given patch size (to downsample masks to patch resolution)
patch_quant_filter = torch.nn.Conv2d(1, 1, PATCH_SIZE, stride=PATCH_SIZE, bias=False)
patch_quant_filter.weight.data.fill_(1.0 / (PATCH_SIZE * PATCH_SIZE))

# image resize transform to dimensions divisible by patch size
def resize_transform(
    image: Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
) -> torch.Tensor:
    w, h = image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(image, (h_patches * patch_size, w_patches * patch_size)))

### Extracting Features
Now we will extract patch features for each of the two images. Each tensor of patch features has shape `[D, H, W]`, where `D` is feature dimensionality, and `H` and `W` are image dimensions in the number of patches.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

patch_features = []
patch_mask_values = []

with torch.inference_mode():
    with torch.autocast(device_type='cuda', dtype=torch.float32):
        for image, mask in tqdm([(image_left, mask_left), (image_right, mask_right)], desc="Processing images"):
            # Process mask (alpha channel) if available
            if mask is not None:
                mask_resized = resize_transform(mask)
                # Quantize mask to patch resolution
                mask_quantized = patch_quant_filter(mask_resized).squeeze().detach().cpu()
                patch_mask_values.append(mask_quantized)
            else:
                patch_mask_values.append(None)
            
            # processing image
            image = image.convert('RGB')
            image_resized = resize_transform(image)
            image_resized = TF.normalize(image_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)
            image_resized = image_resized.unsqueeze(0).cuda()

            # Get features using Hugging Face model
            outputs = model(image_resized, output_hidden_states=True)
            
            # Calculate expected number of patches
            h_patches = image_resized.shape[2] // PATCH_SIZE
            w_patches = image_resized.shape[3] // PATCH_SIZE
            num_patches = h_patches * w_patches
            
            # Get the last hidden state
            # DINOv3 outputs include: [CLS] + [register tokens] + [patch tokens]
            # We need to extract only the patch tokens (last num_patches tokens)
            last_hidden = outputs.last_hidden_state
            patch_tokens = last_hidden[:, -num_patches:, :]  # Take last num_patches tokens
            
            # Reshape to [D, H, W] format
            feats = patch_tokens.squeeze(0).permute(1, 0).view(HIDDEN_DIM, h_patches, w_patches)
            
            patch_features.append(feats.detach().cpu())

dim = HIDDEN_DIM
print(f"Feature shape: {patch_features[0].shape}")
print(f"Mask shape: {patch_mask_values[0].shape if patch_mask_values[0] is not None else 'No mask'}")

### Matching Patches

For each patch in the left image, we search for the closest patch in the right image using [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) between patch features.

In [ ]:
# Threshold for considering a patch as foreground (based on alpha channel)
# Alpha values are normalized to [0, 1], so 0.5 means at least 50% opaque
MASK_FG_THRESHOLD = 0.5

patch_features[0] = F.normalize(patch_features[0], p=2, dim=0)
patch_features[1] = F.normalize(patch_features[1], p=2, dim=0)

heatmaps = torch.einsum(
    "k f, f h w -> k h w",
    patch_features[0].view(dim, -1).permute(1, 0),
    patch_features[1],
)

# compute 2D patch locations in the left image
n_patches_left = patch_features[0].shape[1] * patch_features[0].shape[2]
patch_indices_left = torch.arange(n_patches_left)
locs_2d_left = (
    torch.stack(
        (
            patch_indices_left // patch_features[0].shape[2],  # row
            patch_indices_left % patch_features[0].shape[2]    # column
        ),
        dim=-1
    ) + 0.5
) * PATCH_SIZE

# compute the corresponding 2D patch locations in the right image
patch_indices_right = torch.flatten(heatmaps, start_dim=-2).argmax(dim=-1)
locs_2d_right = (
    torch.stack(
        (
            patch_indices_right // patch_features[1].shape[2],  # row
            patch_indices_right % patch_features[1].shape[2]    # column
        ),
        dim=-1
    ) + 0.5
) * PATCH_SIZE

# Create foreground selection based on alpha masks
if patch_mask_values[0] is not None and patch_mask_values[1] is not None:
    # foreground patches mask in the left image
    patches_left_fg_selection = (patch_mask_values[0].view(-1) > MASK_FG_THRESHOLD)
    # left image patches mask for patches that map to a foreground patch in the right image
    patches_right_fg_selection = (patch_mask_values[1].view(-1)[patch_indices_right] > MASK_FG_THRESHOLD)
    # select foreground left image patches that map to foreground right image patches
    patches_fg_selection = (patches_left_fg_selection * patches_right_fg_selection)
else:
    # No masks available, use all patches
    patches_fg_selection = torch.ones(n_patches_left, dtype=torch.bool)

# select (row, col) coordinates for the matched patches
locs_2d_left_fg = locs_2d_left[patches_fg_selection, :]
locs_2d_right_fg = locs_2d_right[patches_fg_selection, :]

print(f"Total patches: {n_patches_left}, Foreground patches: {patches_fg_selection.sum().item()}")

### Dense Correspondence
We now show dense correspondences between patches using the color space estimated through PCA. Closer colors correspond to more similar patches.

In [ ]:
pca = PCA(n_components=3, whiten=True)
# Fit PCA only on foreground patches
fg_patches_left = patch_features[0].view(dim, -1).permute(1, 0)[patches_fg_selection]
pca.fit(fg_patches_left)
print(f"PCA fitted on {fg_patches_left.shape[0]} foreground patches")

In [ ]:
# get colors for the left image
h_patches_left = patch_features[0].shape[1]
w_patches_left = patch_features[0].shape[2]
x_left = patch_features[0].view(dim, -1).permute(1, 0)
projected_image_left = torch.from_numpy(
    pca.transform(x_left.numpy())
).view(h_patches_left, w_patches_left, 3)
# multiply by 2.0 and pass through a sigmoid to get vibrant colors 
projected_image_left = torch.nn.functional.sigmoid(projected_image_left.mul(2.0)).permute(2, 0, 1)

# get colors for the right image
h_patches_right = patch_features[1].shape[1]
w_patches_right = patch_features[1].shape[2]
x_right = patch_features[1].view(dim, -1).permute(1, 0)
projected_image_right = torch.from_numpy(
    pca.transform(x_right.numpy())
).view(h_patches_right, w_patches_right, 3)
projected_image_right = torch.nn.functional.sigmoid(projected_image_right.mul(2.0)).permute(2, 0, 1)

# Apply foreground masks to visualizations (only show object, not background)
if patch_mask_values[0] is not None:
    projected_image_left *= (patch_mask_values[0] > MASK_FG_THRESHOLD)[None, :, :]
if patch_mask_values[1] is not None:
    projected_image_right *= (patch_mask_values[1] > MASK_FG_THRESHOLD)[None, :, :]

plt.figure(figsize=(4, 2), dpi=300)
plt.subplot(1, 2, 1)
plt.imshow(projected_image_left.permute(1, 2, 0))
plt.title("Left Image, Dense Correspondences", fontsize=5)
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(projected_image_right.permute(1, 2, 0))
plt.title("Right Image, Dense Correspondences", fontsize=5)
plt.axis('off')
plt.show()

### Sparse Correspondence

Finally, we show the established correspondences via the sparse set of matched points.

In [ ]:
# image scale to go from patches to the original image coordinates
scale_left = image_left.height / IMAGE_SIZE
scale_right = image_right.height / IMAGE_SIZE


STRATIFY_DISTANCE_THRESHOLD = 100.0

def compute_distances_l2(X, Y, X_squared_norm, Y_squared_norm):
    distances = -2 * X @ Y.T
    distances.add_(X_squared_norm[:, None]).add_(Y_squared_norm[None, :])
    return distances


def stratify_points(pts_2d: torch.Tensor, threshold: float = 100.0) -> torch.Tensor:
    # pts_2d: [N, 2]
    n = len(pts_2d)
    max_value = threshold + 1
    pts_2d_sq_norms = torch.linalg.vector_norm(pts_2d, dim=1)
    pts_2d_sq_norms.square_()
    distances = compute_distances_l2(pts_2d, pts_2d, pts_2d_sq_norms, pts_2d_sq_norms)
    distances.fill_diagonal_(max_value)
    distances_mask = torch.empty((n, n), dtype=pts_2d.dtype, device=pts_2d.device)
    torch.le(distances, threshold, out=distances_mask)
    ones_vec = torch.ones(n, device=pts_2d.device, dtype=pts_2d.dtype)
    counts_vec = torch.mv(distances_mask, ones_vec)
    indices_mask = np.ones(n)
    while torch.any(counts_vec).item():
        index_max = torch.argmax(counts_vec).item()
        indices_mask[index_max] = 0
        distances[index_max, :] = max_value
        distances[:, index_max] = max_value
        torch.le(distances, threshold, out=distances_mask)
        torch.mv(distances_mask, ones_vec, out=counts_vec)
    indices_to_exclude = np.nonzero(indices_mask == 0)[0]
    indices_to_keep = np.nonzero(indices_mask > 0)[0]
    return indices_to_exclude, indices_to_keep

print(f"Non-stratified foreground points: {tuple(locs_2d_left_fg.shape)}")

indices_to_exclude, indices_to_keep = stratify_points(locs_2d_left_fg * scale_left, STRATIFY_DISTANCE_THRESHOLD**2)

sparse_points_left_yx = locs_2d_left_fg[indices_to_keep, :].cpu().numpy()
sparse_points_right_yx = locs_2d_right_fg[indices_to_keep, :].cpu().numpy()

print(f"Stratified points: {sparse_points_left_yx.shape}")

# show original left and right images
fig = plt.figure(figsize=(20,10))
ax1 = fig.add_subplot(121)
ax1.imshow(image_left)
ax1.set_axis_off()
ax2 = fig.add_subplot(122)
ax2.imshow(image_right)
ax2.set_axis_off()


for i, (row_left, col_left), (row_right, col_right) in zip(
    indices_to_keep, sparse_points_left_yx, sparse_points_right_yx
):
    row_left_orig, col_left_orig = locs_2d_left_fg[i]
    # use the color used for PCA visualization
    color = projected_image_left[
        :,
        int(row_left_orig / PATCH_SIZE),
        int(col_left_orig / PATCH_SIZE)
    ].cpu().numpy()
    con = ConnectionPatch(
        xyA=(col_left * scale_left, row_left * scale_left),
        xyB=(col_right * scale_right, row_right * scale_right),
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color=color,
    )
    ax2.add_artist(con)